# Earth Observation Extraction & Spatial Feature Preprocessing

**Project**: Nepal Flood (Bhote Koshi – Trishuli River Basin)

---

### Overview & Objectives

This notebook establishes an end-to-end Earth Observation (EO) pipeline for multi-temporal flood impact analysis:

1. **Sentinel-1 SAR**: Dual-polarization (VV/VH) backscatter extraction, temporal median compositing (Baseline, Pre-flood, Post-flood, Recovery), and SAR change detection (`dVV`, `dVH`).
2. **Sentinel-2 Optical**: Cloud-filtered surface reflectance compositing, spectral index derivation (NDVI, NDWI, NDBI), and optical change detection (`dNDVI`, `dNDWI`, `dNDBI`).
3. **Topographic Features**: SRTM 30m DEM acquisition and true metric slope computation via UTM Zone 45N projection.
4. **Spatial Alignment**: Precision reprojection to a master reference raster grid (Sentinel-1 30m).
5. **Feature Matrix Construction**: Tabular aggregation of 19 spatial features with exact geographic coordinates (`longitude`, `latitude`).
6. **Ground Truth & External Validation**: Inspection of Copernicus EMS (EMSR927) rapid mapping and OPERA DSWx-S1 availability.
7. **Proxy Target Labeling**: Multi-modal standardized anomaly scoring to generate binary impact labels (`Y`) for downstream machine learning.


## Setup, Environment & Project Paths


In [ ]:
import sys, json, ee, geemap, rasterio
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from rasterio.warp import Resampling

# Resolve project paths dynamically
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import consolidated data preparation helper
import Module.data_prep as dp
time_metadata = dp.load_time_metadata()

DATA_DIR = PROJECT_ROOT / "Data"
RAW_RASTERS_DIR = DATA_DIR / "rasters" / "raw"
TABLES_DIR = DATA_DIR / "tables"
for d in [RAW_RASTERS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data storage : {DATA_DIR}")

## Earth Engine Initialization & Area of Interest (AOI)


In [ ]:
# Initialize Earth Engine and define AOI bounding box for Bhote Koshi - Trishuli
AOI = dp.init_earth_engine(project="astrotourism-darksky")
print("AOI Coordinates:", AOI.coordinates().getInfo())

## Sentinel-1 SAR Processing

---

Extract Interferometric Wide (IW) swath Ground Range Detected (GRD) imagery with dual polarization (VV & VH).  
Generate median composites across Baseline (2025), Pre-flood, Post-flood, and Recovery periods, and compute radar backscatter change (`dVV`, `dVH`).


In [ ]:
# Query Sentinel-1 GRD collection
Sentinel_1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(AOI)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
)

# Generate temporal composites
s1_composites = dp.get_temporal_composites(Sentinel_1, AOI, bands=["VV", "VH"])
S1_baseline = s1_composites["baseline"]
S1_pre = s1_composites["pre"]
S1_post = s1_composites["post"]
S1_recovery = s1_composites["recovery"]

# Compute SAR backscatter change: Post minus Pre
dVV_ee = S1_post.select("VV").subtract(S1_pre.select("VV")).rename("dVV")
dVH_ee = S1_post.select("VH").subtract(S1_pre.select("VH")).rename("dVH")
print("Sentinel-1 composites and change layers created successfully.")

In [ ]:
# Interactive Map Preview using geemap
S1_Map = geemap.Map()
S1_Map.centerObject(AOI, 10)
S1_Map.addLayer(S1_pre.select("VV"), {"min": -25, "max": 5}, "S1 Pre-Flood VV")
S1_Map.addLayer(S1_post.select("VV"), {"min": -25, "max": 5}, "S1 Post-Flood VV")
S1_Map.addLayer(dVV_ee, {"min": -5, "max": 5, "palette": ["0000ff", "ffffff", "ff0000"]}, "dVV Change")
S1_Map

In [ ]:
# Export Sentinel-1 GeoTIFFs to local Data directory
dp.export_geotiff(S1_baseline.select(["VV", "VH"]), RAW_RASTERS_DIR / "S1_2025_baseline.tif", AOI, scale=30)
dp.export_geotiff(S1_pre.select(["VV", "VH"]), RAW_RASTERS_DIR / "S1_pre.tif", AOI, scale=30)
dp.export_geotiff(S1_post.select(["VV", "VH"]), RAW_RASTERS_DIR / "S1_post.tif", AOI, scale=30)


## Sentinel-2 Optical Processing

---

Query Level-2A surface reflectance (`COPERNICUS/S2_SR_HARMONIZED`), filter cloudy scenes (<80%), extract visible, NIR, and SWIR bands (`B3`, `B4`, `B8`, `B11`), and compute:

- **NDVI** (Normalized Difference Vegetation Index): $(B8 - B4) / (B8 + B4)$
- **NDWI** (Normalized Difference Water Index): $(B3 - B8) / (B3 + B8)$
- **NDBI** (Normalized Difference Built-up Index): $(B11 - B8) / (B11 + B8)$
- **Optical Changes**: `dNDVI`, `dNDWI`, `dNDBI`


In [ ]:
# Query Sentinel-2 collection with cloud threshold
Sentinel_2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(AOI)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
)

# Generate temporal composites with key spectral bands
s2_composites = dp.get_temporal_composites(Sentinel_2, AOI, bands=["B3", "B4", "B8", "B11"])
S2_pre = s2_composites["pre"]
S2_post = s2_composites["post"]

# Calculate spectral indices
S2_indices_pre = dp.calculate_optical_indices(S2_pre)
S2_indices_post = dp.calculate_optical_indices(S2_post)

# Calculate optical changes: Post minus Pre
dNDVI_ee = S2_indices_post.select("NDVI").subtract(S2_indices_pre.select("NDVI")).rename("dNDVI")
dNDWI_ee = S2_indices_post.select("NDWI").subtract(S2_indices_pre.select("NDWI")).rename("dNDWI")
dNDBI_ee = S2_indices_post.select("NDBI").subtract(S2_indices_pre.select("NDBI")).rename("dNDBI")

# Stack 9 optical feature bands: pre-indices, post-indices, and differential indices
optical_features_ee = ee.Image.cat([
    S2_indices_pre.select("NDVI").rename("NDVI_pre"),
    S2_indices_pre.select("NDWI").rename("NDWI_pre"),
    S2_indices_pre.select("NDBI").rename("NDBI_pre"),
    S2_indices_post.select("NDVI").rename("NDVI_post"),
    S2_indices_post.select("NDWI").rename("NDWI_post"),
    S2_indices_post.select("NDBI").rename("NDBI_post"),
    dNDVI_ee,
    dNDWI_ee,
    dNDBI_ee
])
print("Optical layers to export:", optical_features_ee.bandNames().getInfo())

In [ ]:
# Export 9-band Sentinel-2 indices and changes
dp.export_geotiff(optical_features_ee, RAW_RASTERS_DIR / "S2_indices_change.tif", AOI, scale=30)


## Digital Elevation Model (SRTM 30m)

---

Extract SRTM GL1 30m topographic elevation to capture valley morphology, flood inundation risk, and landslide susceptibility.


In [ ]:
# Acquire SRTM DEM over AOI
DEM_ee = ee.Image("USGS/SRTMGL1_003").clip(AOI)
dp.export_geotiff(DEM_ee.rename("DEM"), RAW_RASTERS_DIR / "DEM.tif", AOI, scale=30)

# Summary of local downloaded GeoTIFF files
print("\n--- Local Data Directory Inspection ---")
total_mb = 0
for f in sorted(RAW_RASTERS_DIR.glob("*.tif")):
    size_mb = f.stat().st_size / (1024 ** 2)
    total_mb += size_mb
    print(f" - {f.name:<25}: {size_mb:6.2f} MB")
print(f"Total GeoTIFF Storage: {total_mb:.2f} MB")


## Local Spatial Alignment & Metric Slope Calculation

---

All remote sensing data must share the exact same spatial grid (pixel dimensions, affine transform, and CRS).

We set `S1_pre.tif` as the **Master Spatial Reference Grid**.

Furthermore, calculating topographic slope directly in geographic degrees yields severe distortion; we therefore reproject DEM to metric UTM (EPSG:32645, Nepal UTM 45N), compute gradients in meters, and reproject the slope in degrees back to the master grid.


In [ ]:
# Load Master Spatial Reference from S1_pre.tif
with rasterio.open(RAW_RASTERS_DIR / "S1_pre.tif") as src:
    VV_pre = src.read(1).astype("float32")
    VH_pre = src.read(2).astype("float32")
    reference_profile = src.profile.copy()
    reference_transform = src.transform
    reference_crs = src.crs
    reference_shape = (src.height, src.width)

with rasterio.open(RAW_RASTERS_DIR / "S1_post.tif") as src:
    VV_post = src.read(1).astype("float32")
    VH_post = src.read(2).astype("float32")

print(f"Master Reference CRS   : {reference_crs}")
print(f"Master Reference Shape : {reference_shape}")

# Align 9 Sentinel-2 bands to Master Grid
s2_path = DATA_DIR / "S2_indices_change.tif"
s2_aligned_stack = np.stack([
    dp.align_band(s2_path, reference_profile, band=b, resampling=Resampling.bilinear)
    for b in range(1, 10)
])
(NDVI_pre, NDWI_pre, NDBI_pre, NDVI_post, NDWI_post, NDBI_post, dNDVI, dNDWI, dNDBI) = s2_aligned_stack
print(f"Sentinel-2 aligned shape : {s2_aligned_stack.shape}")

# Align DEM and calculate metric slope in degrees (EPSG:32645)
DEM = dp.align_band(DATA_DIR / "DEM.tif", reference_profile, band=1, resampling=Resampling.bilinear)
slope = dp.calculate_metric_slope(DATA_DIR / "DEM.tif", reference_profile, utm_crs="EPSG:32645", resolution=30)

# Calculate local SAR changes
dVV = VV_post - VV_pre
dVH = VH_post - VH_pre

print(f"DEM range   : {np.nanmin(DEM):.1f}m to {np.nanmax(DEM):.1f}m")
print(f"Slope range : {np.nanmin(slope):.1f}\u00b0 to {np.nanmax(slope):.1f}\u00b0")
print(f"dVV range   : {np.nanmin(dVV):.2f} to {np.nanmax(dVV):.2f} dB")


In [ ]:
# Quick visual inspection of SAR VV change
plt.figure(figsize=(9, 6))
plt.imshow(dVV, cmap="RdBu_r", vmin=-5, vmax=5)
plt.colorbar(label="VV Change (dB)")
plt.title("Sentinel-1 VV Backscatter Change (Post - Pre)")
plt.axis("off")
plt.tight_layout()
plt.show()

## Feature Matrix Assembly & Exploratory Data Analysis

---

Assemble all 17 remote sensing features along with pixel coordinate grids (`longitude`, `latitude`) into a structured pandas DataFrame.


In [ ]:
# Dictionary of all spatial feature arrays
arrays_dict = {
    "VV_pre": VV_pre,
    "VH_pre": VH_pre,
    "VV_post": VV_post,
    "VH_post": VH_post,
    "dVV": dVV,
    "dVH": dVH,
    "NDVI_pre": NDVI_pre,
    "NDWI_pre": NDWI_pre,
    "NDBI_pre": NDBI_pre,
    "NDVI_post": NDVI_post,
    "NDWI_post": NDWI_post,
    "NDBI_post": NDBI_post,
    "dNDVI": dNDVI,
    "dNDWI": dNDWI,
    "dNDBI": dNDBI,
    "DEM": DEM,
    "slope": slope
}

# Construct tabular DataFrame with coordinates and finite pixel masking
feature_df, valid_mask = dp.build_feature_table(arrays_dict, reference_transform)

# Save unlabeled feature table
UNLABELED_CSV = TABLES_DIR / "EO_feature_table_unlabeled.csv"
feature_df.to_csv(UNLABELED_CSV, index=False)
print(f"Saved unlabeled feature table: {UNLABELED_CSV} ({feature_df.shape[0]:,} rows, {feature_df.shape[1]} cols)")
feature_df.head()


In [ ]:
# Summary statistics of key change variables
change_vars = ["dVV", "dVH", "dNDVI", "dNDWI", "dNDBI"]
print("--- Change Variables Summary ---")
display(feature_df[change_vars].describe().T)

# 5-panel distribution histograms
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
for ax, var, c in zip(axes, change_vars, colors):
    ax.hist(feature_df[var].dropna(), bins=50, color=c, edgecolor="none", alpha=0.85)
    ax.set_title(var, fontweight="bold")
    ax.set_xlabel("Delta Value")
    ax.set_ylabel("Pixel Count")
plt.tight_layout()
plt.show()

## External Validation Exploration & Proxy Label Generation

---

In rapid disaster scenarios, field labels are scarce. We verify external data availability (Copernicus EMS EMSR927 activation and OPERA DSWx-S1 water extent), then generate a robust multi-modal proxy ground-truth label `Y` based on standardized change anomalies.


In [ ]:
# Query Copernicus EMS Rapid Mapping (EMSR927 Nepal Flood) API
import requests
cems_url = "https://rapidmapping.emergency.copernicus.eu/backend/dashboard-api/public-activations/?code=EMSR927"

try:
    resp = requests.get(cems_url, timeout=20)
    if resp.status_code == 200:
        cems_data = resp.json()
        act = cems_data["results"][0]
        print(f"CEMS Activation: {act.get('code')} - {act.get('name')}")
        print(f"Number of AOIs defined: {len(act.get('aois', []))}")
        for aoi in act.get("aois", []):
            print(f" - AOI {aoi.get('number')}: {aoi.get('name')} ({len(aoi.get('products', []))} products)")
    else:
        print(f"CEMS API responded with status {resp.status_code}")
except Exception as e:
    print(f"Notice: CEMS API query skipped ({e})")

In [ ]:
# Generate Multi-Modal Proxy Target Y
# 1. Standardize (Z-score) key changes: dVV, dVH, dNDVI, dNDWI
# 2. Calculate composite anomaly score: mean(|Z|)
# 3. Set top 20% anomaly threshold as flood/landslide impact class (Y=1)
labeled_df = dp.generate_proxy_labels(
    feature_df,
    change_vars=("dVV", "dVH", "dNDVI", "dNDWI"),
    quantile=0.80
)

# Keep standard feature set + target Y and drop incomplete rows
feature_columns = [
    "longitude", "latitude",
    "VV_pre", "VH_pre",
    "VV_post", "VH_post",
    "dVV", "dVH",
    "NDVI_pre", "NDWI_pre", "NDBI_pre",
    "NDVI_post", "NDWI_post", "NDBI_post",
    "dNDVI", "dNDWI", "dNDBI",
    "DEM", "slope",
    "Y"
]
final_df = labeled_df[feature_columns].dropna().copy()

# Save final labeled dataset
FINAL_LABELED_CSV = TABLES_DIR / "EO_feature_table_labelled.csv"
final_df.to_csv(FINAL_LABELED_CSV, index=False)

print(f"\nSuccessfully saved final dataset: {FINAL_LABELED_CSV}")
print(f"Final Dataset Shape: {final_df.shape[0]:,} rows x {final_df.shape[1]} columns")
print("\nTarget (Y) Class Proportion:")
print(final_df["Y"].value_counts(normalize=True).apply(lambda p: f"{p*100:.2f}%"))
# Automatically sync extracted feature definitions to config.json
config_path = PROJECT_ROOT / "config.json"
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        ml_cfg = json.load(f)
else:
    ml_cfg = {}

ml_cfg.setdefault("FEATURE_GROUPS", {})
s1_feats = [c for c in feature_columns if c in ["VV_pre", "VH_pre", "VV_post", "VH_post", "dVV", "dVH"]]
s2_feats = [c for c in feature_columns if any(c.startswith(p) for p in ["NDVI", "NDWI", "NDBI", "dNDVI", "dNDWI", "dNDBI"])]
terrain_feats = [c for c in feature_columns if c in ["DEM", "slope"]]

ml_cfg["FEATURE_GROUPS"]["S1_FEATURES"] = s1_feats
ml_cfg["FEATURE_GROUPS"]["S2_FEATURES"] = s2_feats
ml_cfg["FEATURE_GROUPS"]["TERRAIN_FEATURES"] = terrain_feats

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(ml_cfg, f, indent=2)

print(f"\nSuccessfully updated {config_path.name} with extracted features:")
print(f" - S1 Features ({len(s1_feats)}): {s1_feats}")
print(f" - S2 Features ({len(s2_feats)}): {s2_feats}")
print(f" - Terrain Features ({len(terrain_feats)}): {terrain_feats}")
